# Kirkwood gaps

Reads one run's `epoch_<step>.bin` files and draws the three views that show
whether a gap opened: where the particles ended up in the $(a, e)$ plane, how
the distribution of $a$ changed, and *when* each part of it changed.

Point `RUN` at another directory to look at a different run.

In [ ]:
import sys

sys.path.insert(0, "python")

import matplotlib.pyplot as plt

from kirkwood_plot import plot_run

RUN = "data/gap_1e5"

fig = plot_run(RUN)
plt.show()

## Depletion at each resonance

Share of the population inside a $\pm 0.025$ AU window around each resonance,
before and after. A ratio near 1.0 means the resonance did nothing; the
`initial` column is the same window in the step-0 file, so it also shows how
much of the apparent signal is just where the window landed.

In [ ]:
import numpy as np

from kirkwood_io import elements, read_run, resonance_axes, semimajor_axis

run = read_run(RUN)
a_initial, _ = elements(run[0])
a_final, e_final = elements(run[-1])

resonances = resonance_axes(
    semimajor_axis(run[0].planet_r[0], run[0].planet_v[0], run[0].G)
)

HALF_WIDTH_AU = 0.025


def share_inside(a, centre):
    """Fraction of the population within HALF_WIDTH_AU of `centre`."""
    return float(np.mean(np.abs(a - centre) < HALF_WIDTH_AU))


print(f"{run[-1].time_years:,.0f} yr, {run[-1].n_alive:,} particles, "
      f"{len(run)} epochs\n")
print(f"{'res':>4} {'a_res':>7} {'initial':>9} {'final':>9} {'ratio':>7}")
for label, axis in sorted(resonances.items(), key=lambda kv: kv[1]):
    before = share_inside(a_initial, axis)
    after = share_inside(a_final, axis)
    print(f"{label:>4} {axis:7.3f} {before:9.4f} {after:9.4f} {after / before:7.3f}")